In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

print("STEP 1: MYANMAR SENTIMENT DATA EXPLORATION (EDA)")


STEP 1: MYANMAR SENTIMENT DATA EXPLORATION (EDA)


In [3]:
df = pd.read_csv('../data/accuracy_test/test.csv')
print('File loaded!')

File loaded!


In [4]:
print("\n   First 5 rows:")
print(df.head())


   First 5 rows:
                                              text  label
0                         Don't like this one much    NaN
1         သီချင်း‌တွေ ခပ်စိပ်စိပ်လုပ်နေတာ မိုက်တယ်    NaN
2                                         cool bro    NaN
3  can't wait to dance at the live performances <3    NaN
4                              Bro this is so good    NaN


In [5]:
print("\n🔍 4. MISSING VALUES CHECK...")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0]
if len(missing_df) > 0:
    print(missing_df)
else:
    print("   ✅ No missing values found!")


🔍 4. MISSING VALUES CHECK...
       Missing  Percentage
text         1    0.089047
label     1123  100.000000


In [9]:
types = df['label'].nunique()
types


0

In [10]:
sentiment_counts = df['label'].value_counts()
print(f"\n   Sentiment distribution (counts):")
for sent, count in sentiment_counts.items():
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)
    print(f"      {sent:15} {bar} {count:4} ({pct:5.1f}%)")


   Sentiment distribution (counts):


In [12]:
df['text'] = df['Text-MM'].astype(str).str.len()
df['text'] = df['Text-MM'].astype(str).str.split().str.len()

KeyError: 'Text-MM'

In [8]:
print("\n🔤 8. ENCODING CHECK (Zawgyi vs Unicode)...")


def detect_zawgyi(text):
    """Simple heuristic to detect Zawgyi encoding"""
    if not isinstance(text, str):
        return False
    zawgyi_chars = ['ဥ', 'ဦ', 'ဧ', 'ဩ', 'ဪ']
    unicode_chars = ['ဉ', 'ည', 'ဋ', 'ဌ', 'ဍ']
    zawgyi_score = sum(1 for c in zawgyi_chars if c in text)
    unicode_score = sum(1 for c in unicode_chars if c in text)
    return zawgyi_score > unicode_score


df['is_zawgyi'] = df['Text-MM'].apply(detect_zawgyi)
zawgyi_count = df['is_zawgyi'].sum()
unicode_count = len(df) - zawgyi_count

print(f"\n   Zawgyi encoded: {zawgyi_count} ({zawgyi_count/len(df)*100:.1f}%)")
print(f"   Unicode encoded: {unicode_count} ({unicode_count/len(df)*100:.1f}%)")

if zawgyi_count > 0:
    print(f"\n   ⚠️  WARNING: Found {zawgyi_count} Zawgyi-encoded texts!")
    print(f"   These need conversion to Unicode before training.")
    print(f"\n   Sample Zawgyi text:")
    zawgyi_sample = df[df['is_zawgyi'] == True]['Text-MM'].iloc[0]
    print(f"      {zawgyi_sample[:100]}")
else:
    print(f"\n   ✅ All texts are in Unicode format!")


🔤 8. ENCODING CHECK (Zawgyi vs Unicode)...


KeyError: 'Text-MM'

In [19]:
print("\n. DUPLICATE CHECK...")

duplicates = df.duplicated(subset=['Text-MM']).sum()
print(f"\n   Exact duplicate texts: {duplicates} ({duplicates/len(df)*100:.1f}%)")

if duplicates > 0:
    print(f"   Found duplicate texts that may need removal.")


. DUPLICATE CHECK...

   Exact duplicate texts: 28 (3.8%)
   Found duplicate texts that may need removal.


In [27]:
if 'Platform' in df.columns:
    print("\n📱 10. PLATFORM DISTRIBUTION...")
    platform_counts = df['Platform'].value_counts()
    for platform, count in platform_counts.head(5).items():
        pct = (count / len(df)) * 100
        print(f"      {platform}: {count} ({pct:.1f}%)")


📱 10. PLATFORM DISTRIBUTION...
       Instagram : 258 (35.2%)
       Facebook : 231 (31.6%)
       Twitter : 128 (17.5%)
       Twitter  : 115 (15.7%)
